# Chem 427, Project #11: Nanoparticle size by three techniques

**Introduction:**
In this project we aim to determine if AFM height reads silica nanoparticles smaller than SEM and SAXS do, while reading gold nanoparticles correctly, by more than the combined uncertainty. Additionally, we aim to determine if our own particle segmentation can reproduce the expert-annotated size distribution.

## Data availability

All data are public and were downloaded on 2026-11-08; the MD5 checksum of each zip matches the one listed on its Zenodo record. All cleaning happens in code below, and the notebook does not download anything.

| Files in `data/` | Content | Source | License |
|---|---|---|---|
| `VSL.zip` | VSL laboratory images as 2D arrays | pollen-metrology.com, *nPSize VSL images as 2D arrays*, Zenodo (2021), https://doi.org/10.5281/zenodo.5578927 | CC0 |
| `LNE1.zip`, `LNE2.zip`, `LNE3.zip` | LNE laboratory images as 2D arrays | pollen-metrology.com, *nPSize LNE images as 2D arrays*, Zenodo (2021), https://doi.org/10.5281/zenodo.5578878 | CC0 |
| `annotations.zip` | Expert particle annotations (`annotations.json`) | pollen-metrology.com, *nPSize JSON file: image annotations*, Zenodo (2021), https://doi.org/10.5281/zenodo.5577401 | CC0 |
| `nPSize*.dat` | SAXS curves | Deumer, J., Gollwitzer, C., Zenodo (2022), https://doi.org/10.5281/zenodo.5886834 | CC BY 4.0 |
| `17NRM04_D3_ValidationReport_v1.0_Accepted.pdf` | Validation report with reference sizes | Bartczak, D., Hodoroaba, V.-D., Zenodo (2022), https://doi.org/10.5281/zenodo.7016466 | CC BY 4.0 |

Accessed 2026-11-08.

## Parameters

Constants that encode a scientific or statistical choice, each with its unit and reason. `ASPECT_RATIO_FLAG` is set from the silica particle shapes (see "Checks on the result" and D4 in `decisions-log.md`). `BOOTSTRAP_UNIT` is the unit each bootstrap resample draws. `GOLD_SPLIT_NM` separates the 30 nm and 60 nm gold populations. Running this cell prints nothing.

In [ ]:
import numpy as np

RNG_SEED = 427              # dimensionless; fixed so bootstrap CIs are identical on every rerun
N_BOOTSTRAP = 10_000        # resamples; enough that the CI endpoints change by < ~0.1 nm between seeds
CI_LEVEL = 0.95             # fraction; conventional two-sided confidence level
K_COVERAGE = 2              # dimensionless; coverage factor for expanded uncertainty (~95 %, GUM)
ASPECT_RATIO_FLAG = 1.12    # dimensionless (max Feret / min Feret); above this an EM particle is flagged as probably
                            # elongated or fused. 1.12 is the 99th percentile in silica TEM 2020 (finest pixels, no tail)
                            # and matches the gap in SEM LNE at ~5.6 nm (≈ 1.12 for a 48 nm particle). Flags only; nothing is removed.
BOOTSTRAP_UNIT = "image"    # resampling unit; particles in one image share the tip or beam, calibration and background,
                            # so they err together (S14: draw whole the unit that errs together). "particle" is the alternative.
GOLD_SPLIT_NM = 44          # nm; gold particles below are the 30 nm population, at or above the 60 nm one. Middle of the gap in
                            # the histogram: 38–50 nm holds 25 of 620 AFM, 5 of 318 SEM and 8 of 422 TEM particles.

rng = np.random.default_rng(RNG_SEED)

## Loading the expert annotations

`annotations.json` (inside `data/annotations.zip`) nests laboratory → microscope → image → particle. Each row of `annotations` below is **one expert-annotated particle in one image**. The columns give its laboratory, technique, material (`npsize`), image name, pixel size (nm/px), AFM height (nm, filled only for AFM), maximum and minimum Feret diameter (nm), and the particle's centroid position in the image (nm, used to check for duplicate particles). Each measurement is stored as a `{unit, value}` pair, so the loader checks that the unit is nm and keeps the value. The cell prints the total row count (expect 11,227) and the count for each lab and technique.

In [ ]:
import json
import zipfile

import pandas as pd

SIZE_FIELDS = {                              # JSON key -> column name (all in nm)
    "afm_height": "afm_height_nm",
    "maximum_feret_diameter": "feret_max_nm",
    "minimum_feret_diameter": "feret_min_nm",
}

with zipfile.ZipFile("data/annotations.zip") as z:
    raw = json.load(z.open("annotations.json"))

rows = []
for lab in raw.values():
    for scope_key, scope in lab.items():
        if not scope_key.startswith("microscope"):    # skip the "name" entry
            continue
        for image in scope["data"].values():
            for particle_key, particle in image["particles"].items():
                if not particle_key.startswith("particle"):   # skip the per-image "total"
                    continue
                row = {
                    "laboratory": lab["name"],
                    "technique": scope["type"],
                    "npsize": image["npsize"],
                    "image": image["name"],
                    "pixel_size_nm": image["pixel_size"],     # nm per pixel
                }
                for key, column in SIZE_FIELDS.items():
                    entry = particle["measurements"][key]
                    if entry["unit"] != "nm":
                        raise ValueError(f"{key} is in {entry['unit']}, not nm")
                    row[column] = entry["value"]
                centroid = particle["measurements"]["centroid"]   # particle position in the image
                if centroid["unit"] != "nm":
                    raise ValueError(f"centroid is in {centroid['unit']}, not nm")
                row["centroid_x_nm"] = centroid["x"]
                row["centroid_y_nm"] = centroid["y"]
                rows.append(row)

annotations = pd.DataFrame(rows)
for column in SIZE_FIELDS.values():
    annotations[column] = pd.to_numeric(annotations[column])   # None -> NaN

print(f"{len(annotations)} annotated particles")
print(annotations.groupby(["laboratory", "technique"]).size())

## Problems found on loading

This cell checks the two materials in scope, nPSize01 (gold) and nPSize10 (silica), for problems that could bias a size comparison, and prints one line per check. It removes nothing. The scientific choice is what counts as a problem: (1) missing size values, (2) pixel size varying between images, since it sets the size resolution, (3) physically impossible values (size ≤ 0 nm, or minimum Feret > maximum Feret), (4) groups with few particles or images, (5) duplicate rows, and (6) format quirks: entries in the JSON that are not microscopes or particles, which the loader skips. A duplicate counts as the same particle only if it has the same image and centroid; identical sizes alone are expected, because sizes are measured in whole pixels. What was done about each: (1) the missing AFM heights are structural, since only AFM measures height, and stay as NaN; (2) the pixel sizes are kept, since every size is already in nm, and differ by laboratory, which is why SEM is resampled within each laboratory in the Result; (3) none found; (4) the small silica AFM group is carried into the uncertainty by the image bootstrap in the Result; (5) nothing removed, since no particle appears twice; (6) skipped by the loader. Outliers are checked in "Checks on the result".

Expect six lines: AFM height missing only on non-AFM rows, a fixed AFM pixel size but several per electron-microscopy group, no impossible values, silica AFM as the smallest group, a nonzero count of identical-size rows but zero same-particle duplicates, and the skipped JSON entries.

In [ ]:
MATERIALS = ["01", "10"]                       # nPSize01 = gold, nPSize10 = silica
size_columns = list(SIZE_FIELDS.values())      # all in nm

in_scope = annotations[annotations["npsize"].isin(MATERIALS)]
groups = in_scope.groupby(["npsize", "technique"])
is_afm = in_scope["technique"] == "AFM"

afm_missing_non_afm = in_scope.loc[~is_afm, "afm_height_nm"].isna().sum()
afm_missing_afm = in_scope.loc[is_afm, "afm_height_nm"].isna().sum()
feret_missing = in_scope[["feret_max_nm", "feret_min_nm"]].isna().sum().sum()
print(f"(1) Missing: afm_height_nm on {afm_missing_non_afm} of {(~is_afm).sum()} non-AFM rows (structural) "
      f"and {afm_missing_afm} AFM rows; Feret diameters on {feret_missing} rows")

pixel_sizes = groups["pixel_size_nm"].agg(["min", "max", "nunique"])            # nm/px
em_pixels = pixel_sizes.drop(index="AFM", level="technique")
afm_pixel = in_scope.loc[is_afm, "pixel_size_nm"].unique()
print(f"(2) Pixel size: AFM fixed at {', '.join(f'{p:.2f}' for p in afm_pixel)} nm/px; electron microscopy "
      f"{em_pixels['nunique'].min()}–{em_pixels['nunique'].max()} values per group, "
      f"{em_pixels['min'].min():.2f} to {em_pixels['max'].max():.2f} nm/px")

nonpositive = int((in_scope[size_columns] <= 0).sum().sum())
min_above_max = int((in_scope["feret_min_nm"] > in_scope["feret_max_nm"]).sum())
print(f"(3) Impossible values: {nonpositive} sizes <= 0 nm, {min_above_max} rows with min Feret > max Feret")

group_counts = groups.agg(n_particles=("image", "size"), n_images=("image", "nunique"))
smallest = group_counts["n_particles"].idxmin()
print(f"(4) Smallest group: nPSize{smallest[0]} {smallest[1]}, {group_counts.loc[smallest, 'n_particles']} particles "
      f"in {group_counts.loc[smallest, 'n_images']} images")

identity_columns = ["laboratory", "technique", "npsize", "image", "pixel_size_nm"]
same_sizes = in_scope.duplicated(subset=identity_columns + size_columns).sum()
same_particle = in_scope.duplicated(subset=["image", "centroid_x_nm", "centroid_y_nm"]).sum()
print(f"(5) Duplicates: {same_sizes} rows with the same image and identical sizes; "
      f"{same_particle} with the same image and centroid (same particle twice)")

non_microscope_keys = sum(not key.startswith("microscope") for lab in raw.values() for key in lab)
images = [image for lab in raw.values() for key, scope in lab.items()
          if key.startswith("microscope") for image in scope["data"].values()]
non_particle_keys = sum(not key.startswith("particle") for image in images for key in image["particles"])
print(f"(6) Format quirks skipped (all files): {non_microscope_keys} non-microscope entries in a laboratory, "
      f"{non_particle_keys} non-particle entries in {len(images)} images")

## Exploration: descriptive statistics

This cell summarises the size distribution of each group, for each material. The scientific choices come from the decisions log:
- **Size measures:** the primary size is the expert AFM height for AFM and the **minimum Feret diameter** for SEM and TEM (D2).
- **Groups:** SEM is the primary reference, pooled over BAM and LNE and also shown per laboratory, so the difference between labs is visible; TEM is shown as context (D1).
- **What is not done yet:** gold (nPSize01) is pooled over its 30 nm and 60 nm populations; it is split in the Result.

Median and IQR are shown because the analysis is median-based, and the mean beside the median shows whether a distribution is skewed. Expect a 10-row table in nm, with silica AFM at about 44 nm (n = 42) and SEM at about 48 nm, and wide IQRs for gold.

In [ ]:
MATERIAL_NAMES = {"01": "gold", "10": "silica"}

summary_rows = []
for material in MATERIALS:
    rows_m = in_scope[in_scope["npsize"] == material]
    sem = rows_m[rows_m["technique"] == "SEM"]
    groups_m = {                                  # group label -> (rows, primary size column)
        "AFM": (rows_m[rows_m["technique"] == "AFM"], "afm_height_nm"),
        "SEM (BAM + LNE)": (sem, "feret_min_nm"),
        "  SEM BAM": (sem[sem["laboratory"] == "BAM"], "feret_min_nm"),
        "  SEM LNE": (sem[sem["laboratory"] == "LNE"], "feret_min_nm"),
        "TEM": (rows_m[rows_m["technique"] == "TEM"], "feret_min_nm"),
    }
    for label, (rows_g, column) in groups_m.items():
        sizes = rows_g[column]                    # nm
        q1, q3 = sizes.quantile([0.25, 0.75])
        summary_rows.append({
            "material": f"{MATERIAL_NAMES[material]} ({material})",
            "group": label,
            "n particles": len(rows_g),
            "n images": rows_g["image"].nunique(),
            "median": sizes.median(),
            "IQR": q3 - q1,
            "mean": sizes.mean(),
        })

size_summary = pd.DataFrame(summary_rows).set_index(["material", "group"])
size_summary.round(1)                             # sizes in nm

For silica, the median AFM height is 43.7 nm (IQR 3.0 nm, n = 42 in 10 images), compared with a median SEM minimum Feret of 47.7 nm (IQR 2.7 nm, n = 287) and 49.6 nm by TEM (n = 353). For gold, the pooled medians are not comparable between groups: BAM's SEM annotations sit almost entirely in the 60 nm population (median 64.3 nm, IQR 6.2 nm), while LNE, TEM and AFM span both populations (IQR 29 to 30 nm).

## Exploration: size distributions

This figure compares the shape of the size distributions of AFM height, SEM minimum Feret (BAM + LNE, the primary reference) and TEM minimum Feret (the secondary reference), with one panel per material. The per-lab split is left out to keep the figure readable; it is in the table above. The scientific choices:
- **Probability density, not counts:** the groups differ in size (n = 42 to 620), so each curve is normalised to unit area.
- **Frequency polygons:** each curve joins the density at each bin centre, so that three overlapping distributions stay readable. Each series has its own line style and marker.
- **Bin width:** 4 nm for gold and 2 nm for silica. Electron-microscopy sizes come in whole-pixel steps of 0.9 to 1.9 nm, and bins narrower than about one step would make a comb-shaped artifact.

Expect two modes near 30 nm and 60 nm for gold, and a single narrow peak near 45 to 50 nm for silica.

In [ ]:
import matplotlib.pyplot as plt

HIST_BIN_WIDTH_NM = {"01": 4.0, "10": 2.0}         # nm; >= ~1 SEM pixel step so whole-pixel Feret values do not make a comb
HIST_RANGE_NM = {"01": (0, 110), "10": (30, 70)}   # nm; covers every in-scope value of the plotted measures

series_style = {                              # label -> (technique, size column, colour, line style, marker)
    "AFM height": ("AFM", "afm_height_nm", "#2a78d6", "-", "o"),
    "SEM min Feret (BAM + LNE)": ("SEM", "feret_min_nm", "#eb6834", "--", "s"),
    "TEM min Feret": ("TEM", "feret_min_nm", "#1baf7a", ":", "^"),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, material in zip(axes, MATERIALS):
    rows_m = in_scope[in_scope["npsize"] == material]
    low, high = HIST_RANGE_NM[material]
    bin_edges = np.arange(low, high + HIST_BIN_WIDTH_NM[material], HIST_BIN_WIDTH_NM[material])   # nm
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2                                            # nm
    for label, (technique, column, colour, linestyle, marker) in series_style.items():
        sizes = rows_m.loc[rows_m["technique"] == technique, column]
        density, _ = np.histogram(sizes, bins=bin_edges, density=True)   # 1/nm
        ax.plot(bin_centers, density, color=colour, linestyle=linestyle, marker=marker,
                markersize=5, linewidth=1.5, label=f"{label} (n = {len(sizes)})")
    ax.set_title(f"{MATERIAL_NAMES[material]} (nPSize{material})")
    ax.set_xlabel("Particle size (nm)")
    ax.set_ylabel("Probability density (1/nm)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

For gold, all three techniques show two well-separated modes near 30 nm and 60 nm, with few particles between 38 and 50 nm (25 of 620 by AFM, 5 of 318 by SEM, 8 of 422 by TEM). For silica, the AFM height peak (median 43.7 nm) lies about 4 nm below the SEM and TEM peaks, and TEM has a tail from 55 to 65 nm that holds 69 of its 353 particles.

## Result: AFM height against SEM minimum Feret

This cell compares the median AFM height with the median SEM minimum Feret diameter (BAM + LNE) and gives the AFM − SEM difference in nm with a bootstrap 95% CI. The scientific choices:
- **Median:** a minority of particles sit on the large side (the elongated electron-microscopy particles in "Checks on the result" are larger than the round ones), and the median is less moved by them than the mean. For silica the mean and median agree within 0.1 nm, so there the choice changes little.
- **Resampling unit:** each resample draws whole images (`BOOTSTRAP_UNIT`), because particles in one image share the tip or beam, the calibration and the background, and so err together. SEM images are drawn within each laboratory, so each resample keeps the share of each lab. AFM and SEM are resampled independently, and the difference is taken resample by resample.
- **Named alternative:** the same bootstrap drawing single particles, which assumes every particle is independent. It is shown beside the image CI.
- **Gold split:** gold (nPSize01) is a mix of 30 nm and 60 nm particles, and the labs annotated different shares of each, so a pooled median compares mixes, not techniques. Each particle is assigned by its primary size (AFM height or SEM minimum Feret) to the 30 nm population if below `GOLD_SPLIT_NM` and to the 60 nm population otherwise, and each population is compared on its own. The cutoff sits in a gap that holds few particles in every technique, so a nearby cutoff would move only a few particles. BAM annotated no particles below the cutoff, so the 30 nm SEM reference is LNE alone; the `SEM labs` column shows this.

Expect three rows (silica, gold 30 nm, gold 60 nm): the particle and image counts, the SEM labs, the medians and difference in nm, the image CI and the particle CI. For silica, expect a difference of about −4 nm.

In [ ]:
def resampling_units(rows, column, unit):
    """Split one group's sizes (nm) into strata (laboratories), each a list of resampling units.

    unit = "particle": every particle is its own unit; unit = "image": all particles of one image form a unit.
    """
    strata = []
    for _, rows_lab in rows.groupby("laboratory"):
        if unit == "image":
            strata.append([rows_img[column].to_numpy() for _, rows_img in rows_lab.groupby("image")])
        else:
            strata.append([np.array([size]) for size in rows_lab[column]])
    return strata


def bootstrap_medians(strata, n_boot, rng):
    """Median (nm) of each of n_boot resamples; units are drawn with replacement within each stratum."""
    medians = np.empty(n_boot)
    for b in range(n_boot):
        picked = [units[i] for units in strata for i in rng.integers(0, len(units), len(units))]
        medians[b] = np.median(np.concatenate(picked))
    return medians


gold = in_scope[in_scope["npsize"] == "01"]
gold_size = gold["afm_height_nm"].where(gold["technique"] == "AFM", gold["feret_min_nm"])   # nm, primary size
comparisons = {                                   # label -> rows compared
    "silica (10)": in_scope[in_scope["npsize"] == "10"],
    f"gold (01), 30 nm (< {GOLD_SPLIT_NM} nm)": gold[gold_size < GOLD_SPLIT_NM],
    f"gold (01), 60 nm (>= {GOLD_SPLIT_NM} nm)": gold[gold_size >= GOLD_SPLIT_NM],
}
ALTERNATIVE_UNIT = "particle" if BOOTSTRAP_UNIT == "image" else "image"
tail = (1 - CI_LEVEL) / 2

result_rows = []
for label, rows_c in comparisons.items():
    afm = rows_c[rows_c["technique"] == "AFM"]
    sem = rows_c[rows_c["technique"] == "SEM"]    # BAM + LNE, primary reference (D1)
    row = {
        "comparison": label,
        "AFM n (images)": f"{len(afm)} ({afm['image'].nunique()})",
        "SEM n (images)": f"{len(sem)} ({sem['image'].nunique()})",
        "SEM labs": " + ".join(sorted(sem["laboratory"].unique())),
        "AFM median (nm)": afm["afm_height_nm"].median(),
        "SEM median (nm)": sem["feret_min_nm"].median(),
    }
    row["AFM − SEM (nm)"] = row["AFM median (nm)"] - row["SEM median (nm)"]
    for unit in (BOOTSTRAP_UNIT, ALTERNATIVE_UNIT):
        afm_boot = bootstrap_medians(resampling_units(afm, "afm_height_nm", unit), N_BOOTSTRAP, rng)
        sem_boot = bootstrap_medians(resampling_units(sem, "feret_min_nm", unit), N_BOOTSTRAP, rng)
        low, high = np.quantile(afm_boot - sem_boot, [tail, 1 - tail])     # nm
        row[f"{unit} CI low (nm)"] = low
        row[f"{unit} CI high (nm)"] = high
    result_rows.append(row)

result_table = pd.DataFrame(result_rows).set_index("comparison")
result_table.round(2)

*Your sentence giving the result: the difference, its CI, and whether the CI excludes zero; then how the particle CI compares.*

## Checks on the result

**Outliers: a particle shape flag (D4).** A particle seen by electron microscopy is flagged if its aspect ratio (maximum Feret / minimum Feret, dimensionless) exceeds `ASPECT_RATIO_FLAG`, as a probable elongated or fused particle. The scientific choice is a physical flag on shape instead of a statistical fence on size. The cutoff is the 99th percentile of the aspect ratio in the TEM 2020 session, which has the finest pixels (0.26 nm/px) and no large-size tail, so it shows how round single particles of this silica are. AFM particles are not flagged, because tip dilation distorts lateral shape, so the AFM row is blank in the flag columns. No particle is removed; the table shows how far each silica median moves if the flagged particles are left out.

**TEM sessions.** The CEA TEM images come from three sessions, read from the year at the start of each image name. The cell prints the median minimum Feret of each, to show whether one session differs as a whole, which a per-particle flag cannot detect.

Expect two printed lines, then a 3-row table in which the flagged particles are a sizeable fraction of each electron-microscopy group. The flag is applied to silica only so far; gold is not checked yet.

In [ ]:
silica = in_scope[in_scope["npsize"] == "10"]
silica_em = silica[silica["technique"].isin(["SEM", "TEM"])].copy()
silica_em["aspect_ratio"] = silica_em["feret_max_nm"] / silica_em["feret_min_nm"]   # dimensionless
tem = silica_em[silica_em["technique"] == "TEM"]
tem_session = tem["image"].str[:4]                # session = year at the start of the image name

reference_p99 = tem.loc[tem_session == "2020", "aspect_ratio"].quantile(0.99)
print(f"Aspect ratio, 99th percentile, TEM 2020: {reference_p99:.3f} (ASPECT_RATIO_FLAG = {ASPECT_RATIO_FLAG})")
session_medians = tem.groupby(tem_session)["feret_min_nm"].median()                  # nm
print("TEM median min Feret by session (nm):", ", ".join(f"{s} {m:.1f}" for s, m in session_medians.items()))

flag_groups = {                                   # label -> (rows, primary size column)
    "AFM": (silica[silica["technique"] == "AFM"], "afm_height_nm"),
    "SEM (BAM + LNE)": (silica_em[silica_em["technique"] == "SEM"], "feret_min_nm"),
    "TEM (all sessions)": (tem, "feret_min_nm"),
}
flag_rows = []
for label, (rows_g, column) in flag_groups.items():
    row = {"group": label, "n particles": len(rows_g), "median (nm)": rows_g[column].median()}
    if "aspect_ratio" in rows_g:
        flagged = rows_g["aspect_ratio"] > ASPECT_RATIO_FLAG
        row["n flagged"] = int(flagged.sum())
        row["median without flagged (nm)"] = rows_g.loc[~flagged, column].median()
        row["change (nm)"] = row["median without flagged (nm)"] - row["median (nm)"]
    else:                                         # AFM: lateral shape is tip-dilated, so no shape flag
        row.update({"n flagged": np.nan, "median without flagged (nm)": np.nan, "change (nm)": np.nan})
    flag_rows.append(row)

flag_table = pd.DataFrame(flag_rows).set_index("group")
flag_table["n flagged"] = flag_table["n flagged"].astype("Int64")   # whole counts; blank for AFM
flag_table.round(2)                               # nm, except counts

*Your sentence giving the numbers: how many particles are flagged, how far the SEM median moves, and what the TEM sessions show.*

## Next steps

*Yours to write: what these data cannot answer, and the assumption you check next and how (for example, whether the expert `afm_height` is the maximum height above the substrate, checked against the VSL height maps).*

## Environment

This cell prints the Python version and the versions of the packages used above, so the results can be reproduced. Expect Python 3.14 and the versions locked in `uv.lock`.

In [ ]:
import sys
import matplotlib

print(f"Python {sys.version.split()[0]}")
for module in (np, pd, matplotlib):
    print(f"{module.__name__} {module.__version__}")